In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/adsb_data/bronze/live_states")

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType, BooleanType,
    IntegerType, ArrayType, DateType
)

expected_schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("time_position", LongType(), True),
    StructField("last_contact", LongType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("baro_altitude", DoubleType(), True),
    StructField("on_ground", BooleanType(), True),
    StructField("velocity", DoubleType(), True),
    StructField("true_track", DoubleType(), True),
    StructField("vertical_rate", DoubleType(), True),
    StructField("sensors", ArrayType(IntegerType()), True),
    StructField("geo_altitude", DoubleType(), True),
    StructField("squawk", StringType(), True),
    StructField("spi", BooleanType(), True),
    StructField("position_source", IntegerType(), True),
    StructField("category", IntegerType(), True),
    StructField("api_timestamp", LongType(), True),
    StructField("ingested_at", StringType(), True),
    StructField("ingest_date", DateType(), True),
    StructField("ingest_hour", IntegerType(), True),
])

assert df.schema == expected_schema, f"Schema mismatch: {df.schema}"

In [0]:
#this code is mainly for testing (the data colleected when dev didn't contain all the conditions we'd hope to see so we create mock data for these events for donwstream)
from datetime import date

mock_rows = [
    # Aircraft abc123 - starts on ground, then takes off
    ("abc123", "BAW123", "United Kingdom", 1775058950, 1775058950, -0.4614, 51.4775, 0.0,    True,  0.0,   0.0,  0.0,  None, 0.0,  "1234", False, 0, 0, 1775058950, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("abc123", "BAW123", "United Kingdom", 1775058980, 1775058980, -0.4614, 51.4775, 0.0,    True,  0.0,   0.0,  0.0,  None, 0.0,  "1234", False, 0, 0, 1775058980, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("abc123", "BAW123", "United Kingdom", 1775059010, 1775059010, -0.4614, 51.4775, 1000.0, False, 250.0, 90.0, 10.0, None, 950.0, "1234", False, 0, 0, 1775059010, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("abc123", "BAW123", "United Kingdom", 1775059040, 1775059040, -0.4614, 51.4775, 2000.0, False, 250.0, 90.0, 10.0, None, 1900.0,"1234", False, 0, 0, 1775059040, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),

    # Aircraft abc123 - gap > 600 seconds
    ("abc123", "BAW123", "United Kingdom", 1775059770, 1775059770, -0.3,   51.5,   5000.0, False, 250.0, 90.0, 0.0,  None, 4900.0,"1234", False, 0, 0, 1775059770, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("abc123", "BAW123", "United Kingdom", 1775059800, 1775059800, -0.3,   51.5,   5000.0, False, 250.0, 90.0, 0.0,  None, 4900.0,"1234", False, 0, 0, 1775059800, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),

    # Aircraft xyz999 - always airborne
    ("xyz999", "AFR456", "France",         1775058950, 1775058950, 2.5478, 49.0097, 8000.0, False, 300.0, 180.0, 0.0, None, 7900.0,"5678", False, 0, 0, 1775058950, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("xyz999", "AFR456", "France",         1775058980, 1775058980, 2.5478, 49.0097, 8000.0, False, 300.0, 180.0, 0.0, None, 7900.0,"5678", False, 0, 0, 1775058980, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("xyz999", "AFR456", "France",         1775059010, 1775059010, 2.5478, 49.0097, 8000.0, False, 300.0, 180.0, 0.0, None, 7900.0,"5678", False, 0, 0, 1775059010, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),

    # Aircraft flx001 - long enough flight to pass the filter (10+ states, 5+ minutes)
    ("flx001", "FLX001", "France", 1775058950, 1775058950, 2.5478,  49.0097, 0.0,    True,  0.0,   0.0,  0.0,   None, 0.0,    "9999", False, 0, 0, 1775058950, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059010, 1775059010, 2.5478,  49.0097, 0.0,    True,  0.0,   0.0,  0.0,   None, 0.0,    "9999", False, 0, 0, 1775059010, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059070, 1775059070, 1.8000,  49.5000, 1000.0, False, 250.0, 90.0, 10.0,  None, 950.0,  "9999", False, 0, 0, 1775059070, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059130, 1775059130, 1.2000,  50.0000, 3000.0, False, 280.0, 90.0, 10.0,  None, 2900.0, "9999", False, 0, 0, 1775059130, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059190, 1775059190, 0.8000,  50.3000, 5000.0, False, 300.0, 90.0, 5.0,   None, 4900.0, "9999", False, 0, 0, 1775059190, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059250, 1775059250, 0.4000,  50.6000, 7000.0, False, 310.0, 90.0, 0.0,   None, 6900.0, "9999", False, 0, 0, 1775059250, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059310, 1775059310, 0.1000,  50.9000, 8000.0, False, 320.0, 90.0, 0.0,   None, 7900.0, "9999", False, 0, 0, 1775059310, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059370, 1775059370, -0.1000, 51.1000, 8000.0, False, 320.0, 90.0, 0.0,   None, 7900.0, "9999", False, 0, 0, 1775059370, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059430, 1775059430, -0.2000, 51.2000, 7000.0, False, 310.0, 90.0, -3.0,  None, 6900.0, "9999", False, 0, 0, 1775059430, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059490, 1775059490, -0.3000, 51.3000, 6000.0, False, 300.0, 90.0, -5.0,  None, 5900.0, "9999", False, 0, 0, 1775059490, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059550, 1775059550, -0.4000, 51.4000, 3000.0, False, 280.0, 90.0, -10.0, None, 2900.0, "9999", False, 0, 0, 1775059550, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
    ("flx001", "FLX001", "France", 1775059610, 1775059610, -0.4614, 51.4775, 0.0,    True,  0.0,   0.0,  0.0,   None, 0.0,    "9999", False, 0, 0, 1775059610, "2025-01-01T00:00:00Z", date(2025, 1, 1), 0),
]

mock_df = spark.createDataFrame(mock_rows, schema=expected_schema)
df = df.union(mock_df)

In [0]:
df = df.dropDuplicates(['icao24', "api_timestamp"]).na.drop(subset=['longitude', 'latitude','icao24'])


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

window_spec = Window.partitionBy("icao24").orderBy("api_timestamp")
df = df.withColumn("prev_onground",lag("on_ground",1).over(window_spec)).\
        withColumn("prev_api_timestamp", lag("api_timestamp",1).over(window_spec))

In [0]:
from pyspark.sql.functions import when

df = df.withColumn("is_new_flight", 
                   when(col("prev_api_timestamp").isNull(), True) #prev_api_timestamp first record of the flight 
                   .when((col("on_ground") == False) & (col("prev_onground") == True) , True) #takeoff 
                   .when(col("api_timestamp") - col("prev_api_timestamp") > 600, True).otherwise(False)) 
                   # we stopped receiving for about 10min 600sec could be for various e.g.
                   # Flight landed 
                   # Aircraft went out of ADS-B receiver range 
                   # Aircraft turned off its transponder
                   # Data collection gap on OpenSky's side

In [0]:
from pyspark.sql.functions import sum as spark_sum

cumulative_window = window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)#include the first row
df = df.withColumn("segment",spark_sum(col("is_new_flight").cast("int")).over(cumulative_window))

In [0]:
from pyspark.sql.functions import concat, lit
#flightid = icao24_segment e.g c0ffee_3 ( flight coffee segement 3)
df = df.withColumn('flight_id', concat(col("icao24"), lit("_"), col("segment")))

In [0]:
from pyspark.sql.functions import min, max, avg, count, first, last, min_by, max_by, to_date, from_unixtime

silver_flight_df = df.groupBy("flight_id") \
  .agg(
    first("icao24").alias("icao24"),
    first("callsign").alias("callsign"),
    min("api_timestamp").alias("departure_ts"),
    max("api_timestamp").alias("arrival_ts"),
    avg("baro_altitude").alias("avg_altitude"),
    max("baro_altitude").alias("max_altitude"),
    avg("velocity").alias("avg_speed"),
    max("velocity").alias("max_speed"),
    first("origin_country").alias("origin_country"),
    min_by("latitude", "api_timestamp").alias("origin_lat"),
    min_by("longitude", "api_timestamp").alias("origin_lon"),
    max_by("latitude", "api_timestamp").alias("dest_lat"),
    max_by("longitude", "api_timestamp").alias("dest_lon"),
    count("segment").alias("state_count")
  ).withColumn("duration_min", (col("arrival_ts") - col("departure_ts")) / 60 
  ).withColumn("flight_date", to_date(from_unixtime(col("departure_ts"))))


In [0]:
import sys
sys.path.insert(0, "/Volumes/workspace/default/adsb_data/utils")
from geo_utils import haversine_distance, nearest_airport

silver_flight_df = silver_flight_df.withColumn("origin_airport", nearest_airport(col("origin_lat"),col("origin_lon")))\
        .withColumn("dest_airport", nearest_airport(col("dest_lat"),col("dest_lon")))\
        .withColumn("traveled_distance", haversine_distance(col("origin_lat"), col("origin_lon"), col("dest_lat"), col("dest_lon")))


In [0]:
silver_flight_df = silver_flight_df.filter((col("duration_min") >= 5) & (col("state_count") >= 10))

In [0]:
from delta.tables import DeltaTable

SILVER_FLIGHTS_TABLE = "workspace.default.silver_flights"

if spark.catalog.tableExists(SILVER_FLIGHTS_TABLE):
    DeltaTable.forName(spark, SILVER_FLIGHTS_TABLE) \
        .alias("target") \
        .merge(silver_flight_df.alias("source"), "target.flight_id = source.flight_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    silver_flight_df.write \
        .format("delta") \
        .partitionBy("flight_date") \
        .saveAsTable(SILVER_FLIGHTS_TABLE)